# Lab 09 Prelab Walkthrough — Dictionaries, Array Stacking, and Curve Reading

Work through this notebook **before** your Lab 09 session, running each cell
and reading the explanations. It teaches only the *new* Python skills Lab 09
uses, with small simulated datasets. Report the **CHECKPOINT** values in the
Prelab 09 quiz on Canvas.

Everything here runs without hardware.

## Skill 1 — The servo-PWM arithmetic (pencil first, code second)

Lab 09 commands a motor's ESC with a pulse-width signal: a pulse every
20 ms whose width is $u = 1000 + 10\times(\text{throttle \%})$ microseconds.
The ADS pattern generator builds it from two integers:

- a **divider** that turns the 100 MHz base clock into 1 µs ticks
- a **counter** pair: output low for `20000 − u` ticks, high for `u` ticks

Compute them yourself for a 42% throttle command, then check:

In [ ]:
base_clock = 100_000_000          # Hz, ADS pattern generator clock

pct = 42
us = int(1000 + 10 * pct)         # pulse width in microseconds

divider = base_clock // 1_000_000 # ticks of base clock per 1 us tick
low  = 20_000 - us
high = us

print(f"divider = {divider}, low = {low}, high = {high}")
print(f"frame = {(low + high)/1000} ms  (should be 20 ms)")

**CHECKPOINT 1:** For a **67%** throttle command, what are `high` and
`low`? Report both in the Canvas quiz.

## Skill 2 — Dictionaries: datasets with names

You know lists (`runs[0]`) and arrays. A **dictionary** stores values under
*names* instead of positions — and analysis code that reads
`data['PropA']` instead of `results[0]` documents itself.

In [ ]:
import numpy as np

# a dictionary is built with braces, or grown one key at a time
motor = {}
motor['name'] = 'test-motor'
motor['kv'] = 3800
motor['mass_g'] = 11.5

print(motor['kv'])          # read a value back by its key
print(list(motor.keys()))   # what's in here?

In [ ]:
# values can be anything — including arrays, or other dictionaries
data = {}
data['PropA'] = {'throttle': np.array([10, 20, 30]),
                 'thrust':   np.array([0.05, 0.19, 0.40])}
data['PropB'] = {'throttle': np.array([10, 20, 30]),
                 'thrust':   np.array([0.08, 0.25, 0.51])}

for prop in data:                       # looping visits the keys
    print(prop, '30% thrust =', data[prop]['thrust'][-1], 'N')

**CHECKPOINT 2:** Add `data['PropC']` with thrust array
`[0.06, 0.22, 0.45]` (same throttle values) and report
`data['PropC']['thrust'].mean()` to 4 decimals.

In lab, this exact structure holds your two propellers' sweep data —
`data[prop]['T']`, `data[prop]['I']`, and their statistics.

## Skill 3 — `np.stack` and `axis`: statistics over repeated runs

Lab 09 repeats each sweep three times. Each run is an array; `np.stack`
piles them into one higher-dimension array, and then every statistic you
need is a single **axis reduction**. Simulated example — three noisy
"runs" of the same 5-setpoint measurement:

In [ ]:
rng = np.random.default_rng(42)      # seeded: everyone gets the same data

true_values = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
runs = [true_values + rng.normal(0, 0.05, size=5) for _ in range(3)]

arr = np.stack(runs)                 # shape (3, 5): axis 0 = run, axis 1 = setpoint
print(arr.shape)
print(arr.round(3))

In [ ]:
# axis=0 collapses ACROSS RUNS -> one number per setpoint
means = arr.mean(axis=0)
stds  = arr.std(axis=0, ddof=1)
print('means:', means.round(4))
print('stds :', stds.round(4))

# the 95% CI on the mean, exactly as in lab (n = 3 -> df = 2)
from scipy import stats
CI = stats.t.ppf(0.975, df=2) * stds / np.sqrt(3)
print('CIs  :', CI.round(4))

- `arr.mean(axis=0)` — "collapse the run axis": average the 3 runs at
  each setpoint. `axis=1` would instead average the 5 setpoints within
  each run — almost never what you want here. When unsure, check the
  result's shape.
- `[expr for _ in range(3)]` is a **list comprehension** — a one-line
  `for` loop that builds a list. Lab 09 uses one to load three CSVs.

**CHECKPOINT 3:** Report the CI value at the *first* setpoint
(`CI[0]`) to 4 decimals.

## Skill 4 — `np.interp`: reading a curve backwards

Your design task: "what throttle gives 60 g?" — you know the *y*, you want
the *x*. `np.interp(y_target, y_curve, x_curve)` linearly interpolates,
provided `y_curve` is increasing (a thrust curve is).

In [ ]:
throttle = np.array([10, 20, 30, 40, 50, 60, 70])          # %
thrust_g = np.array([5.0, 19.0, 40.0, 68.0, 103.0, 141.0, 185.0])

target = 60.0                                # grams needed
delta_h = np.interp(target, thrust_g, throttle)
print(f"hover throttle = {delta_h:.2f} %")

# sanity check the other way: what thrust does that throttle give?
print(f"check: {np.interp(delta_h, throttle, thrust_g):.1f} g")

**CHECKPOINT 4:** Using the same arrays, what throttle produces
**120 g**? Report to 2 decimals.

## Skill 5 — `np.meshgrid` and 3-D surfaces: evaluating a design space

One `np.interp` call answers one design question. `np.meshgrid` lets the
same arithmetic answer *every combination* of two design variables at
once — in lab, that becomes a flight-time surface over (vehicle mass ×
battery capacity).

In [ ]:
x = np.array([1.0, 2.0, 3.0])        # imagine: mass choices
y = np.array([10.0, 20.0])           # imagine: battery choices
X, Y = np.meshgrid(x, y)

print('X =\n', X)                    # x-value of every combination
print('Y =\n', Y)                    # y-value of every combination
Z = X * Y                            # any formula now works elementwise
print('Z =\n', Z)

In [ ]:
import matplotlib.pyplot as plt

xf = np.linspace(0, 4, 40)
yf = np.linspace(0, 4, 40)
X, Y = np.meshgrid(xf, yf)
Z = np.sin(X) * np.cos(Y) + X        # a stand-in "design function"

fig = plt.figure(figsize=(6, 4.5))
ax = fig.add_subplot(projection='3d')     # 3-D axes
ax.plot_surface(X, Y, Z, cmap='viridis', edgecolor='none')
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('Z')
plt.show()

print(f"Z at (x=2.0, y=1.0): {Z[10, 20]:.4f}")   # note: Z[row=y, col=x]

Note the indexing order in that last line: `meshgrid` puts **y on the
rows and x on the columns**, so `Z[i, j]` is at `yf[i]`, `xf[j]` — the
classic 3-D-plot gotcha.

**CHECKPOINT 5:** Report `Z[10, 20]` to 4 decimals.

## Summary — where each skill appears in lab

| Skill | Where you'll use it in Lab 09 |
|-------|-------------------------------|
| PWM arithmetic | `set_throttle()` — commanding the ESC |
| dictionaries | `data[prop]` — organizing two propellers' sweeps |
| list comprehension | loading the three run CSVs per prop |
| `np.stack` + `axis=0` | thrust/current means and CIs per setpoint |
| `np.interp` | Part-6: finding the hover throttle from your curve |
| `np.meshgrid` + `plot_surface` | Part-6: the endurance design map |
